# 06 — SQL Business Analysis
**Objective:** Answer business questions in MySQL using joins, CTEs, subqueries, window functions.
**Approach:** Load the raw CSVs into MySQL, apply the valid-sale rule in a view, and reconcile
results against the Python numbers from notebook 5 (98,198 orders, R$15,738,221.95).
**Deliverable:** SQL query collection (`sql/`) and business findings.

In [1]:
# Cell 1: import library
import os
from dotenv import load_dotenv
from urllib.parse import quote_plus
from sqlalchemy import create_engine, text
from pathlib import Path
import pandas as pd

In [2]:
# Cell 2: load and connect to MySQL
PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
SQL_DIR = PROJECT_ROOT / "sql"
load_dotenv(PROJECT_ROOT / ".env")
user = os.getenv("MYSQL_USER")
pwd = quote_plus(os.getenv("MYSQL_PASSWORD"))
host = os.getenv("MYSQL_HOST")
port = os.getenv("MYSQL_PORT")
db = os.getenv("MYSQL_DB")
engine = create_engine(f"mysql+pymysql://{user}:{pwd}@{host}:{port}/{db}")
with engine.connect() as conn:
    print("MySQL version:", conn.execute(text("SELECT VERSION()")).scalar())
    print("Current database:", conn.execute(text("SELECT DATABASE()")).scalar())

MySQL version: 8.0.43
Current database: olist_ecommerce


In [3]:
# Cell 3: load the CSVs
import time

tables = {   # geolocation last, because it is slow and unused by the SQL queries
    "customers": ("customers.csv", 99441),
    "orders": ("orders.csv", 99441),
    "order_items": ("order_items.csv", 112650),
    "order_payments": ("order_payments.csv", 103886),
    "order_reviews": ("order_review.csv", 99224),
    "products": ("products.csv", 32951),
    "sellers": ("sellers.csv", 3095),
    "category_translation": ("product_category_name.csv", 71),
    "geolocation": ("geolocation.csv", 1000163),
}
date_cols = {
    "orders": ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
               "order_delivered_customer_date", "order_estimated_delivery_date"],
    "order_items": ["shipping_limit_date"],
    "order_reviews": ["review_creation_date", "review_answer_timestamp"],
}

for t, (f, expected) in tables.items():
    with engine.connect() as conn:
        current = conn.execute(text(f"SELECT COUNT(*) FROM {t}")).scalar()
    if current >= expected:
        print(f"Skipped {t:22s} already has {current:,} rows")
        continue

    start = time.time()
    with engine.begin() as conn:
        conn.execute(text(f"TRUNCATE TABLE {t}"))
    df = pd.read_csv(RAW_DIR / f)
    if t == "products":
        df = df.rename(columns={"product_name_lenght": "product_name_length",
                                "product_description_lenght": "product_description_length"})
    for c in date_cols.get(t, []):
        df[c] = pd.to_datetime(df[c])
    df.to_sql(t, engine, if_exists="append", index=False, chunksize=5000)
    print(f"Loaded  {t:22s} {len(df):>10,} rows in {time.time() - start:.0f}s")

with engine.begin() as conn:
    conn.execute(text("""INSERT IGNORE INTO category_translation VALUES
        ('portateis_cozinha_e_preparadores_de_alimentos', 'portable_kitchen_food_preparers'),
        ('pc_gamer', 'pc_gamer')"""))

Skipped customers              already has 99,441 rows
Skipped orders                 already has 99,441 rows
Skipped order_items            already has 112,650 rows
Skipped order_payments         already has 103,886 rows
Skipped order_reviews          already has 99,224 rows
Skipped products               already has 32,951 rows
Skipped sellers                already has 3,095 rows
Skipped category_translation   already has 73 rows
Skipped geolocation            already has 1,000,163 rows


In [4]:
# Cell 4: helper that runs a .sql

def run_sql_file(name):
    sql = (SQL_DIR / name).read_text(encoding="utf-8")
    with engine.begin() as conn:
        conn.execute(text("SET SESSION MAX_EXECUTION_TIME = 120000"))
        for stmt in sql.split(";"):
            lines = stmt.strip().splitlines()
            body = "\n".join(l for l in lines if not l.strip().startswith("--")).strip()
            if not body:
                continue
            title = lines[0].lstrip("- ").strip() if lines and lines[0].startswith("--") else ""
            result = conn.execute(text(body))
            if result.returns_rows:
                if title:
                    print(f"\n>>> {title}")
                display(pd.DataFrame(result.fetchall(), columns=list(result.keys())))

In [5]:
# Cell 5: category_translation must have 73 rows
with engine.connect() as conn:
    print(conn.execute(text("SELECT COUNT(*) FROM category_translation")).scalar())  # expect 73

73


In [6]:
# Cell 6: data quality
run_sql_file("04_data_quality_checks.sql")


>>> Data quality check --


,table_name,row_count
0,customers,99441
1,geolocation,1000163
2,orders,99441
3,order_items,112650
4,order_payments,103886
5,order_reviews,99224
6,products,32951
7,sellers,3095
8,category_translation,73



>>> Delivered orders with no delivery date (expect 8)


,delivered_no_date
0,8



>>> Orders without items, by status (expect 775 in total)


,order_status,orders_without_items
0,unavailable,603
1,canceled,164
2,created,5
3,invoiced,2
4,shipped,1



>>> Orders with more than one review (expect 547)


,orders_with_multiple_reviews
0,547



>>> Payment total vs items plus freight (expect 98665 compared, 98276 exact, 249 over R$1)


,orders_compared,exact_match,mismatch_over_1
0,98665,98362,249


In [7]:
# Cell 7: revenue
run_sql_file("05_revenue_analysis.sql")


>>> Headline KPIs for valid sales (expect 98198 orders, revenue 15738221.95, AOV 160.27)


,valid_orders,unique_customers,total_revenue,avg_order_value
0,98198,94982,15738221.95,160.27



>>> Revenue of all paid orders vs valid sales (expect about 16008872 vs 15738222)


,revenue_all_orders,revenue_valid_sales,revenue_excluded
0,16008872.12,15738221.95,270650.17



>>> Monthly revenue with month-over-month growth and running total (Jan 2017 to Aug 2018)


,order_month,orders,revenue,aov,mom_pct,running_revenue
0,2017-01,787,137006.76,174.09,None,137006.76
1,2017-02,1718,283621.94,165.09,107.01,420628.70
2,2017-03,2617,425656.40,162.65,50.08,846285.10
3,2017-04,2377,405988.38,170.80,-4.62,1252273.48
4,2017-05,3640,582926.16,160.14,43.58,1835199.64
5,2017-06,3205,499827.47,155.95,-14.26,2335027.11
6,2017-07,3946,578858.58,146.70,15.81,2913885.69
7,2017-08,4272,662071.77,154.98,14.38,3575957.46
8,2017-09,4227,717195.22,169.67,8.33,4293152.68
9,2017-10,4547,764785.28,168.20,6.64,5057937.96


... (2 outputs left out)


>>> Orders and revenue by weekday (Monday = 0)


,weekday_no,weekday,orders,revenue
0,0,Monday,15933,2572843.80
1,1,Tuesday,15700,2511057.82
2,2,Wednesday,15311,2443253.98
3,3,Thursday,14524,2320090.24
4,4,Friday,13904,2262872.36
5,5,Saturday,10725,1734864.63
6,6,Sunday,11808,1841486.24



>>> Revenue by customer state with share of total (top 10)


,customer_state,orders,revenue,revenue_share_pct
0,SP,41124,5878702.01,37.35
1,RJ,12697,2115956.44,13.44
2,MG,11496,1843460.37,11.71
3,RS,5415,877620.20,5.58
4,PR,4982,794407.35,5.05
5,SC,3599,608104.32,3.86
6,BA,3344,607041.45,3.86
7,DF,2120,351350.03,2.23
8,GO,1998,340626.24,2.16
9,ES,2018,323105.75,2.05


In [8]:
# Cell 8: customers (also creates v_latest_review)
run_sql_file("06_customer_analysis.sql")


>>> Orders per customer


,orders_per_customer,customers
0,1,92095
1,2,2651
2,3,188
3,4,29
4,5,9
5,6,5



>>> Repeat purchase rate


,customers,repeat_customers,repeat_rate_pct
0,94982,2887,3.04



>>> Top 10 customers by total spend


,customer_unique_id,customer_state,orders,total_spend,spend_rank
0,0a0a92112bd4c708ca5fde585afaa872,RJ,1,13664.08,1
1,da122df9eeddfedc1dc1f5349a1a690c,RJ,2,7571.63,2
2,763c8b1c9c68a0229c42c9fc6f662b93,ES,1,7274.88,3
3,dc4802a71eae9be1dd28f5d788ceb526,MS,1,6929.31,4
4,459bef486812aa25204be022145caa62,ES,1,6922.21,5
5,ff4159b92c40ebe40454e3e6a7c35ed6,SP,1,6726.66,6
6,4007669dec559734d6f53e029e360987,MG,1,6081.54,7
7,eebb5dda148d3893cdaf5b5ca3040ccb,SP,1,4764.34,8
8,48e1ac109decbb87765a3eade6854098,PB,1,4681.78,9
9,c8460e4251689ba205045f3ea17884a1,RS,4,4655.91,10


... (2 outputs left out)


>>> Days between first and second order for repeat customers


,repeat_customers,avg_days_to_second_order,min_days,max_days
0,2887,80.8,0,609



>>> Payment type mix for valid orders


,payment_type,orders,payment_value,share_pct,avg_installments
0,credit_card,75614,12349453.71,78.47,3.51
1,boleto,19535,2826476.09,17.96,1.00
2,voucher,3745,349874.40,2.22,1.00
3,debit_card,1515,212417.75,1.35,1.00


In [9]:
# Cell 9: materialize the two views as indexed tables (speeds up files 07-09)
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS valid_orders_t"))
    conn.execute(text("DROP TABLE IF EXISTS latest_review_t"))
    conn.execute(text("""
        CREATE TABLE valid_orders_t (
          PRIMARY KEY (order_id),
          INDEX idx_vo_ts (order_purchase_timestamp),
          INDEX idx_vo_state (customer_state)
        ) AS SELECT * FROM v_valid_orders"""))
    conn.execute(text("CREATE TABLE latest_review_t (PRIMARY KEY (order_id)) AS SELECT * FROM v_latest_review"))
with engine.connect() as conn:
    n1 = conn.execute(text("SELECT COUNT(*) FROM valid_orders_t")).scalar()
    n2 = conn.execute(text("SELECT COUNT(*) FROM latest_review_t")).scalar()
print("valid_orders_t :", n1, "PASS" if n1 == 98198 else "FAIL (expect 98198)")
print("latest_review_t:", n2, "PASS" if n2 == 98673 else "FAIL (expect 98673)")

valid_orders_t : 98198 PASS
latest_review_t: 98673 PASS


In [10]:
# Cell 10: switch files 07, 08, 09 from views to tables (backup kept once)
import shutil
for f in ["07_product_analysis.sql", "08_seller_analysis.sql", "09_delivery_analysis.sql"]:
    p = SQL_DIR / f
    bak = p.with_suffix(".sql.bak")
    if not bak.exists():
        shutil.copy(p, bak)
    txt = p.read_text(encoding="utf-8")
    txt = txt.replace("v_valid_orders", "valid_orders_t").replace("v_latest_review", "latest_review_t")
    p.write_text(txt, encoding="utf-8")
    print(f"{f}: old names left = {txt.count('v_valid_orders') + txt.count('v_latest_review')}, "
          f"table names = {txt.count('valid_orders_t') + txt.count('latest_review_t')}")

07_product_analysis.sql: old names left = 0, table names = 7
08_seller_analysis.sql: old names left = 0, table names = 5
09_delivery_analysis.sql: old names left = 0, table names = 10


In [11]:
# Cell 11: products
run_sql_file("07_product_analysis.sql")


>>> Top 10 categories by product revenue (expect health_beauty 1255560.16 first)


,category,items_sold,orders,revenue,avg_price,avg_freight,revenue_rank
0,health_beauty,9631,8799,1255560.16,130.37,18.89,1
1,watches_gifts,5970,5604,1198185.21,200.70,16.77,2
2,bed_bath_table,11097,9399,1035964.06,93.36,18.42,3
3,sports_leisure,8590,7673,979740.92,114.06,19.50,4
4,computers_accessories,7781,6654,904322.02,116.22,18.84,5
5,furniture_decor,8298,6425,727465.05,87.67,20.75,6
6,housewares,6915,5847,626825.80,90.65,21.00,7
7,cool_stuff,3779,3616,620770.49,164.27,22.05,8
8,auto,4204,3872,586585.73,139.53,21.89,9
9,garden_tools,4328,3505,481009.94,111.14,22.76,10



>>> Number of categories that make up 80 percent of revenue (expect 18 of 74)


,total_categories,categories_for_80pct_revenue
0,74,18



>>> Category average review score (latest review, at least 200 orders)


,category,orders,avg_review
0,office_furniture,1262,3.49
1,fixed_telephony,211,3.71
2,home_confort,395,3.83
3,audio,346,3.84
4,unknown,1425,3.86
5,bed_bath_table,9295,3.90
6,furniture_living_room,415,3.91
7,furniture_decor,6375,3.92
8,home_construction,487,3.94
9,telephony,4154,3.95



>>> Best category in each customer state by revenue (top 3 states)


,customer_state,category,revenue,category_rank
0,MG,health_beauty,157433.32,1
1,MG,bed_bath_table,129487.18,2
2,MG,watches_gifts,123473.54,3
3,RJ,watches_gifts,183931.50,1
4,RJ,bed_bath_table,147785.43,2
5,RJ,health_beauty,144912.93,3
6,SP,bed_bath_table,477794.20,1
7,SP,health_beauty,460339.12,2
8,SP,watches_gifts,431119.29,3



>>> Monthly revenue of the top 3 categories (Jan 2017 to Aug 2018)


,order_month,category,revenue
0,2017-01,health_beauty,12561.32
1,2017-01,watches_gifts,8086.52
2,2017-01,bed_bath_table,3960.16
3,2017-02,health_beauty,22838.79
4,2017-02,bed_bath_table,16200.83
5,2017-02,watches_gifts,11756.21
6,2017-03,health_beauty,25927.35
7,2017-03,bed_bath_table,25773.02
8,2017-03,watches_gifts,24970.38
9,2017-04,bed_bath_table,24347.69


In [12]:
# Cell 12: sellers
run_sql_file("08_seller_analysis.sql")


>>> Sellers with 30 or more delivered orders (expect 627 of 2970 sellers)


,sellers_30_plus_orders
0,627



>>> Sellers with the highest late rate (30 or more orders)


,seller_id,seller_state,orders,late_rate_pct,avg_delay_days,revenue
0,ede0c03645598cdfc63ca8237acbe73d,SP,43,34.04,-3.48,2352.15
1,54965bbe3e4f07ae045b90b0b8541f52,PR,73,32.10,-1.69,10351.70
2,ad781527c93d00d89a11eecd9dcad7c1,SP,38,31.58,-8.31,6039.63
3,835f0f7810c76831d6c7d24c7a646d4d,SP,42,31.25,-4.97,4990.20
4,2a1348e9addc1af5aaa619b1a3679d6b,MG,48,29.41,1.35,2697.40
5,66e0557ecc2b4dbea057e93f215f68d8,SP,30,26.67,-8.13,12141.00
6,71039d19d4303bf9054d69e9a9236699,SP,35,26.32,-6.55,3537.20
7,6039e27294dc75811c0d8a39069f52c0,SP,63,25.68,-6.37,13500.80
8,a49928bcdf77c55c6d6e05e09a9b4ca5,SP,96,25.00,-3.81,8646.90
9,ef990a83bbea832f36ebe81376335aa8,SC,43,25.00,-7.81,4642.59



>>> Sellers with late rate above 20 percent (expect 23)


,eligible_sellers,sellers_above_20pct,avg_late_rate
0,627,23,7.66



>>> Top 10 sellers by revenue with their late rate and cumulative revenue share


,seller_id,orders,revenue,late_rate_pct,revenue_rank,cumulative_share_pct
0,4869f7a5dfa277a7dca6462dcf3b52b2,1124,226987.93,11.59,1,1.72
1,53243585a1d6dc2643021fd1853d8905,348,217940.44,4.00,2,3.37
2,4a3ca9315b744ce9f8e9374361493884,1772,196882.12,10.98,3,4.85
3,fa1c13f2614d7b5c4749cbc52fecda94,578,190917.14,10.19,4,6.30
4,7c67e1448b00f6e969d365cea6b010ab,973,186570.05,9.59,5,7.71
5,7e93a43ef30c4f03f38b393420bc753a,319,165981.49,5.59,6,8.97
6,da8622b14eb17ae2831f4ac5b9dab84a,1311,159816.87,7.30,7,10.17
7,7a67c85e85bb2ce8582c35f2203ad736,1145,139658.69,5.89,8,11.23
8,1025f0e2d44d7041d6cf58b6550e0bfa,910,138208.56,9.23,9,12.28
9,955fee9216a65b617aa5c0531780ce60,1261,131836.71,8.08,10,13.27



>>> Seller performance by seller state (states with 100 or more orders)


,seller_state,orders,sellers,late_rate_pct
0,MA,389,1,23.63
1,SP,68635,1769,8.52
2,RJ,4227,163,8.11
3,DF,808,30,6.68
4,ES,310,22,6.59
5,PR,7511,335,6.42
6,SC,3603,184,5.88
7,MG,7733,236,5.55
8,BA,550,18,5.45
9,MT,136,4,4.86


In [13]:
# Cell 13: delivery
run_sql_file("09_delivery_analysis.sql")


>>> Delivery overview (expect 96470 orders, about 8.11 percent late, about 12.56 days)


,delivered_orders,late_rate_pct,avg_delivery_days,avg_estimated_days
0,96469,8.11,12.56,23.74



>>> Review score for late vs on-time orders (expect about 4.29 on time and 2.57 late)


,delivery_status,reviewed_orders,avg_review,one_star_pct,five_star_pct
0,on time or early,88163,4.29,6.59,62.44
1,late,7660,2.57,46.20,22.23



>>> Review score by delay bucket (days versus the estimated date)


,delay_bucket,orders,avg_review,bad_review_pct
0,1. early by 7+ days,70929,4.32,8.97
1,2. early by 0 to 7 days,17234,4.20,10.24
2,3. late by up to 7 days,4409,3.18,36.09
3,4. late by more than 7 days,3251,1.73,78.44


... (2 outputs left out)


>>> Monthly late rate with 3-month moving average (Jan 2017 to Aug 2018)


,order_month,orders,late_rate_pct,moving_avg_3m
0,2017-01,750,3.07,3.07
1,2017-02,1653,3.21,3.14
2,2017-03,2546,5.58,3.95
3,2017-04,2303,7.86,5.55
4,2017-05,3545,3.61,5.68
5,2017-06,3135,3.86,5.11
6,2017-07,3872,3.43,3.64
7,2017-08,4193,3.32,3.54
8,2017-09,4150,5.20,3.98
9,2017-10,4478,5.29,4.60



>>> Extreme deliveries (more than 60 days, expect 306 in total)


,orders_over_60_days,avg_review
0,306,2.14


In [14]:
# Cell 13a: which delivered order is in orders but not in valid_orders_t?
q = """
SELECT o.order_id, o.order_status, o.order_delivered_customer_date,
       (SELECT COUNT(*) FROM order_payments p WHERE p.order_id = o.order_id) AS payment_rows,
       (SELECT COUNT(*) FROM order_items i WHERE i.order_id = o.order_id) AS item_rows
FROM orders o
LEFT JOIN valid_orders_t v ON v.order_id = o.order_id
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
  AND v.order_id IS NULL
"""
display(pd.read_sql(text(q), engine))

,order_id,order_status,order_delivered_customer_date,payment_rows,item_rows
0,bfbd0f9bdef84302105ad712db648a6c,delivered,2016-11-09 07:47:38,0,3


## Business findings

**Scope:** valid sales only (98,198 orders, R$15,738,221.95). SQL results reconcile with the Python numbers from notebook 5, except delivered orders: SQL counts 96,469 against 96,470 in Python (one delivered order has no payment record, see Cell 13a).

### Products
- **Revenue is spread over a few categories.** health_beauty leads with R$1,255,560.16 (8,799 orders), ahead of watches_gifts (R$1,198,185.21) and bed_bath_table (R$1,035,964.06). 18 of 74 categories make up 80% of revenue.
- **Volume and revenue are different rankings.** bed_bath_table sells the most items (11,097) but ranks third on revenue because its average price is low (R$93.36). watches_gifts has an average price of R$200.70 and ranks second on 5,970 items.
- **Weakest reviews (200+ orders):** office_furniture scores lowest at 3.49 (1,262 orders). bed_bath_table scores 3.90 on 9,295 orders, which matters because of its volume.
- **The top 3 states share the same top 3 categories, in a different order:** SP is led by bed_bath_table (R$477,794.20), RJ by watches_gifts (R$183,931.50) and MG by health_beauty (R$157,433.32).
- **Monthly trend:** watches_gifts peaks in Nov 2017 (R$96,575.58) and in May 2018 (R$123,872.66). health_beauty is the largest category in Aug 2018 (R$120,475.71).

### Sellers
- 627 of 2,970 sellers have 30+ delivered orders. 23 of them (3.7%) have a late rate above 20%. The average late rate among the 627 is 7.66%.
- **Revenue is not concentrated:** the top 10 sellers produce 13.27% of revenue, and the largest earns R$226,987.93 (1.72%). The top 10 sellers' late rates range from 4.00% to 11.59%.
- By seller state, SP carries most of the volume (68,635 orders, 1,769 sellers, 8.52% late). MA shows 23.63% late, but it has one seller, so it reflects one seller and not the state.

### Delivery
- 96,469 delivered orders: 8.11% late, 12.56 days on average against 23.74 days estimated.
- **Late delivery drives reviews.** On-time orders average 4.29 stars (6.59% one-star). Late orders average 2.57 (46.20% one-star). Orders more than 7 days late average 1.73 and 78.44% get a bad review.
- **Regional gap:** AL has the highest late rate (23.93%, average delivery 24.54 days), then MA (19.67%, 21.57 days) and PI (15.97%, 19.46 days). SP is best among large states at 8.76 days and 5.89% late. AM has the longest average delivery (26.43 days) but a low late rate (4.14%), so its delivery estimates are long.
- **Monthly late rate** peaks in Nov 2017 (14.31%), Feb 2018 (15.99%) and Mar 2018 (21.36%), then falls to 1.36% in Jun 2018.
- 306 orders took more than 60 days, with an average review of 2.14.

### Limits
These are descriptive results. They show associations (late delivery with low reviews), not causes. The statistical testing comes in notebook 7.